# Week 6 EXERCISE: Customer Support Ticket Priority Classifier

## Overview

Build a **fine-tuning pipeline** for classifying customer support tickets into priority levels.

**Use Case:** Help support teams triage incoming tickets efficiently by automatically classifying them as **Urgent**, **High**, **Medium**, or **Low** priority.

## Key Learnings from Week 6

- **Day 1-2**: Data curation and preprocessing for ML/fine-tuning
- **Day 3**: Evaluation metrics and baselines
- **Day 4**: Deep learning approaches
- **Day 5**: Fine-tuning frontier models (OpenAI GPT)

## Pipeline Steps

1. **Dataset** - Synthetic support tickets with priority labels
2. **Train/Val/Test Split** - Prepare data for fine-tuning
3. **Zero-shot Baseline** - Evaluate frontier model without fine-tuning
4. **Export JSONL** - Prepare training data for fine-tuning
5. **Gradio UI** - Interactive ticket classifier with evaluation

---

In [1]:
# Imports

import os
import json
import random
import re
from collections import Counter
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr

load_dotenv(override=True)

True

In [2]:
# Configuration

OPENROUTER_BASE_URL = "https://openrouter.ai/api/v1"
FRONTIER_MODEL = "openai/gpt-4.1-mini"

# Try OpenRouter first, fall back to OpenAI
api_key = os.getenv("OPENROUTER_API_KEY")
if api_key:
    client = OpenAI(base_url=OPENROUTER_BASE_URL, api_key=api_key)
    print("Using OpenRouter API")
else:
    client = OpenAI()
    FRONTIER_MODEL = "gpt-4.1-mini"
    print("Using OpenAI API directly")

Using OpenRouter API


## Dataset: Customer Support Tickets

Synthetic support ticket examples with priority classifications:

| Priority | Description |
|----------|-------------|
| **Urgent** | System down, security breach, data loss, production outage |
| **High** | Major feature broken, payment issues, significant user impact |
| **Medium** | Feature request, minor bug, performance concern |
| **Low** | General inquiry, documentation question, cosmetic issue |

In [3]:
# Priority levels
PRIORITIES = ("Urgent", "High", "Medium", "Low")

# Synthetic support ticket dataset
TICKETS = [
    # URGENT - System down, security, data loss
    {"ticket": "CRITICAL: Our entire production database is down. No users can access the application. This is affecting 50,000+ customers right now. Need immediate assistance.", "priority": "Urgent"},
    {"ticket": "SECURITY ALERT: We detected unauthorized access to our admin panel. Someone from an unknown IP accessed sensitive customer data. Need to lock down immediately.", "priority": "Urgent"},
    {"ticket": "Data corruption detected in customer payment records. Transactions from the last 24 hours may be affected. This is a compliance issue.", "priority": "Urgent"},
    {"ticket": "Complete service outage. API returning 500 errors for all endpoints. Our mobile app is completely unusable. Revenue impact: $10K/hour.", "priority": "Urgent"},
    {"ticket": "RANSOMWARE ALERT: Our servers have been encrypted. Attackers demanding payment. All systems locked. Business completely halted.", "priority": "Urgent"},
    {"ticket": "Production server crashed and won't restart. Auto-recovery failed. All customer-facing services are offline. SLA breach imminent.", "priority": "Urgent"},
    {"ticket": "Critical vulnerability discovered in authentication system. Attackers can bypass login. Patching required immediately before exploitation.", "priority": "Urgent"},
    {"ticket": "Payment processing completely broken. All transactions failing. Customers cannot complete purchases. E-commerce site effectively down.", "priority": "Urgent"},
    {"ticket": "DDoS attack in progress. Our infrastructure is being overwhelmed. Website unreachable for last 30 minutes. Need mitigation NOW.", "priority": "Urgent"},
    {"ticket": "Backup system failed during restore. Lost 48 hours of customer data. Critical business data unrecoverable without help.", "priority": "Urgent"},
    {"ticket": "SSL certificate expired on production. Chrome showing security warning to all users. Customers afraid to enter payment info.", "priority": "Urgent"},
    {"ticket": "Database replication failed. Primary and secondary out of sync. Risk of data loss if primary fails. Need immediate intervention.", "priority": "Urgent"},
    
    # HIGH - Major features broken, payment issues, significant impact
    {"ticket": "User login is failing for about 30% of our customers. They're getting 'invalid credentials' even with correct passwords. Started 2 hours ago.", "priority": "High"},
    {"ticket": "Credit card payments are being declined randomly. Some customers charged twice. Others not charged but order confirmed. Very confusing.", "priority": "High"},
    {"ticket": "Email notifications stopped working completely. Customers not receiving order confirmations, password resets, or any transactional emails.", "priority": "High"},
    {"ticket": "Search functionality broken on our e-commerce site. Returns no results for any query. Customers can't find products.", "priority": "High"},
    {"ticket": "Mobile app crashes immediately on launch for iOS 17 users. About 40% of our user base affected. App store reviews tanking.", "priority": "High"},
    {"ticket": "Checkout process broken. Users can add items to cart but 'Place Order' button does nothing. No error message shown.", "priority": "High"},
    {"ticket": "User dashboard showing wrong data. Customers seeing other users' information. Privacy concern and potential GDPR issue.", "priority": "High"},
    {"ticket": "API rate limiting is too aggressive. Legitimate integrations being blocked. Partners complaining they can't sync data.", "priority": "High"},
    {"ticket": "File upload feature broken. Users trying to upload documents get timeout errors. Critical for our document management workflow.", "priority": "High"},
    {"ticket": "Subscription renewal failing for annual plans. Customers being downgraded to free tier unexpectedly. Billing team overwhelmed.", "priority": "High"},
    {"ticket": "Two-factor authentication not sending codes. Users locked out of accounts. Password reset also requires 2FA. Catch-22 situation.", "priority": "High"},
    {"ticket": "Report generation timing out for large datasets. Enterprise customers can't generate monthly reports. Contract SLA at risk.", "priority": "High"},
    {"ticket": "Webhook deliveries failing. Integration partners not receiving events. Their automations broken due to our issue.", "priority": "High"},
    {"ticket": "User permissions not applying correctly. Some users have admin access who shouldn't. Security review needed urgently.", "priority": "High"},
    
    # MEDIUM - Feature requests, minor bugs, performance
    {"ticket": "Page load time increased from 2s to 5s after last update. Not critical but users noticing slowdown. Would like investigation.", "priority": "Medium"},
    {"ticket": "Feature request: Can we add dark mode to the dashboard? Many users asking for this. Would improve user experience.", "priority": "Medium"},
    {"ticket": "Export to PDF not formatting tables correctly. Data is there but alignment is off. Low priority but would be nice to fix.", "priority": "Medium"},
    {"ticket": "Calendar integration with Google Calendar showing events 1 hour off. Timezone issue probably. Can work around by adjusting.", "priority": "Medium"},
    {"ticket": "Search results could be more relevant. Looking for ways to improve our search algorithm. Not broken, just could be better.", "priority": "Medium"},
    {"ticket": "Mobile app battery usage seems high. Users reporting drain. Would like optimization but app still functional.", "priority": "Medium"},
    {"ticket": "Feature request: Add bulk edit capability for user management. Currently have to edit one by one. Would save admin time.", "priority": "Medium"},
    {"ticket": "Charts on analytics dashboard not responsive on tablet. Data visible but layout breaks. Desktop and mobile work fine.", "priority": "Medium"},
    {"ticket": "Notification sounds not working on Android. Visual notifications work. Minor issue but some users prefer audio alerts.", "priority": "Medium"},
    {"ticket": "Auto-save feature sometimes takes 10+ seconds. Usually instant. Intermittent issue, can't reproduce consistently.", "priority": "Medium"},
    {"ticket": "Feature request: Would like to customize email templates. Currently using default designs. Branding consistency important.", "priority": "Medium"},
    {"ticket": "Memory usage gradually increases over time. Have to refresh browser after 4-5 hours. Memory leak suspected.", "priority": "Medium"},
    {"ticket": "Could you add keyboard shortcuts for common actions? Power users would appreciate this productivity feature.", "priority": "Medium"},
    {"ticket": "Drag and drop occasionally doesn't register first attempt. Need to try 2-3 times. Annoying but not blocking.", "priority": "Medium"},
    {"ticket": "Date picker defaults to US format. We're UK-based and would prefer DD/MM/YYYY. Localization request.", "priority": "Medium"},
    {"ticket": "Performance degrades with 1000+ items in a list. Pagination would help but not critical for most users.", "priority": "Medium"},
    
    # LOW - General inquiries, documentation, cosmetic
    {"ticket": "Where can I find documentation for the API? Looking for endpoint reference. No urgency, just getting started.", "priority": "Low"},
    {"ticket": "Typo on the pricing page. 'Busines' instead of 'Business'. Minor cosmetic issue.", "priority": "Low"},
    {"ticket": "What's the difference between Pro and Enterprise plans? Looking to potentially upgrade in the future.", "priority": "Low"},
    {"ticket": "The favicon looks slightly blurry on retina displays. Very minor visual thing. Not affecting functionality.", "priority": "Low"},
    {"ticket": "Is there a way to change the accent color in the UI? Brand colors are blue but your default is purple.", "priority": "Low"},
    {"ticket": "How do I export my data if I want to cancel? Just asking for future reference, not canceling now.", "priority": "Low"},
    {"ticket": "Documentation example code is in Python but I use JavaScript. Request for JS examples would be helpful.", "priority": "Low"},
    {"ticket": "Logo in footer is the old version. We rebranded 6 months ago. Would be nice to update when convenient.", "priority": "Low"},
    {"ticket": "What browsers do you officially support? Just curious about your compatibility matrix.", "priority": "Low"},
    {"ticket": "The 'Learn More' link on the homepage goes to a page with not much info. Maybe add more content?", "priority": "Low"},
    {"ticket": "Font size in the footer is a bit small. Hard to read but not critical information anyway.", "priority": "Low"},
    {"ticket": "Do you have a referral program? Would like to recommend your service to colleagues if there's a benefit.", "priority": "Low"},
    {"ticket": "Inconsistent button styles on different pages. Some rounded, some square. Minor design inconsistency.", "priority": "Low"},
    {"ticket": "How long do you retain deleted data? Privacy policy wasn't clear on retention period.", "priority": "Low"},
    {"ticket": "Would be nice to have a changelog page showing recent updates. Just a suggestion for transparency.", "priority": "Low"},
    {"ticket": "Empty state message could be friendlier. 'No data' is accurate but cold. UX writing improvement.", "priority": "Low"},
    {"ticket": "Do you offer student discounts? I'm learning to use your platform for my thesis project.", "priority": "Low"},
    {"ticket": "The tutorial video is from 2022. UI has changed since then. Update when you have time.", "priority": "Low"},
]

print(f"Total tickets in dataset: {len(TICKETS)}")
print(f"Priority distribution: {Counter(t['priority'] for t in TICKETS)}")

Total tickets in dataset: 60
Priority distribution: Counter({'Low': 18, 'Medium': 16, 'High': 14, 'Urgent': 12})


## Train/Validation/Test Split

Split the dataset with stratification to maintain priority distribution.

In [4]:
# Shuffle and split (70% train, 15% val, 15% test)
random.seed(42)
shuffled = TICKETS.copy()
random.shuffle(shuffled)

n = len(shuffled)
n_train = int(0.7 * n)
n_val = int(0.15 * n)

train_data = shuffled[:n_train]
val_data = shuffled[n_train:n_train + n_val]
test_data = shuffled[n_train + n_val:]

print(f"Train: {len(train_data)} | Val: {len(val_data)} | Test: {len(test_data)}")
print(f"\nTrain distribution: {Counter(t['priority'] for t in train_data)}")
print(f"Test distribution: {Counter(t['priority'] for t in test_data)}")

Train: 42 | Val: 9 | Test: 9

Train distribution: Counter({'Low': 13, 'Medium': 12, 'High': 11, 'Urgent': 6})
Test distribution: Counter({'Urgent': 3, 'High': 3, 'Low': 2, 'Medium': 1})


## Zero-Shot Baseline

Evaluate the frontier model's classification ability without any fine-tuning.

In [5]:
# System prompt for classification
SYSTEM_PROMPT = """You are a customer support ticket triage system.
Classify the support ticket into exactly one priority level:
- Urgent: System down, security breach, data loss, production outage
- High: Major feature broken, payment issues, significant user impact
- Medium: Feature request, minor bug, performance concern
- Low: General inquiry, documentation question, cosmetic issue

Respond with ONLY the priority level (Urgent, High, Medium, or Low), nothing else."""

def predict_baseline(ticket: str) -> str:
    """Zero-shot classification via OpenRouter/OpenAI."""
    try:
        response = client.chat.completions.create(
            model=FRONTIER_MODEL,
            messages=[
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user", "content": ticket}
            ],
            max_tokens=10,
            temperature=0
        )
        if response.choices:
            raw = (response.choices[0].message.content or "").strip()
            # Extract priority from response
            for label in PRIORITIES:
                if label.lower() in raw.lower():
                    return label
            return raw or "Medium"
        return "Medium"
    except Exception as e:
        print(f"Prediction error: {e}")
        return "Medium"

# Test single prediction
test_ticket = test_data[0]
prediction = predict_baseline(test_ticket["ticket"])
print(f"Sample ticket: {test_ticket['ticket'][:100]}...")
print(f"True: {test_ticket['priority']} | Predicted: {prediction}")

Prediction error: Error code: 400 - {'error': {'message': 'Provider returned error', 'code': 400, 'metadata': {'raw': '{\n  "error": {\n    "message": "Invalid \'max_output_tokens\': integer below minimum value. Expected a value >= 16, but got 10 instead.",\n    "type": "invalid_request_error",\n    "param": "max_output_tokens",\n    "code": "integer_below_min_value"\n  }\n}', 'provider_name': 'Azure', 'is_byok': False}}, 'user_id': 'user_39l2mYQ8CncrKVDueHpsmmP8E63'}
Sample ticket: Would be nice to have a changelog page showing recent updates. Just a suggestion for transparency....
True: Low | Predicted: Medium


In [6]:
def calculate_accuracy(predictor, data, verbose=False):
    """Calculate accuracy on a dataset."""
    correct = 0
    results = []
    
    for item in data:
        pred = predictor(item["ticket"])
        is_correct = pred == item["priority"]
        if is_correct:
            correct += 1
        results.append({
            "ticket": item["ticket"][:80] + "...",
            "true": item["priority"],
            "predicted": pred,
            "correct": is_correct
        })
        if verbose:
            status = "OK" if is_correct else "WRONG"
            print(f"[{status}] True: {item['priority']:6} | Pred: {pred:6} | {item['ticket'][:60]}...")
    
    accuracy = correct / len(data) if data else 0.0
    return accuracy, results

# Evaluate baseline on test set
print("Evaluating baseline model on test set...\n")
baseline_accuracy, baseline_results = calculate_accuracy(predict_baseline, test_data, verbose=True)
print(f"\n=== Baseline Accuracy: {baseline_accuracy:.1%} ===")
print("This is the number to beat with fine-tuning!")

Evaluating baseline model on test set...

Prediction error: Error code: 400 - {'error': {'message': 'Provider returned error', 'code': 400, 'metadata': {'raw': '{\n  "error": {\n    "message": "Invalid \'max_output_tokens\': integer below minimum value. Expected a value >= 16, but got 10 instead.",\n    "type": "invalid_request_error",\n    "param": "max_output_tokens",\n    "code": "integer_below_min_value"\n  }\n}', 'provider_name': 'Azure', 'is_byok': False}}, 'user_id': 'user_39l2mYQ8CncrKVDueHpsmmP8E63'}
[WRONG] True: Low    | Pred: Medium | Would be nice to have a changelog page showing recent update...
Prediction error: Error code: 400 - {'error': {'message': 'Provider returned error', 'code': 400, 'metadata': {'raw': '{\n  "error": {\n    "message": "Invalid \'max_output_tokens\': integer below minimum value. Expected a value >= 16, but got 10 instead.",\n    "type": "invalid_request_error",\n    "param": "max_output_tokens",\n    "code": "integer_below_min_value"\n  }\n}', 'pr

## Prepare Training Data (JSONL)

Export training and validation data in JSONL format for fine-tuning.

In [7]:
def messages_for(item):
    """Create training message format: user = ticket, assistant = priority."""
    return [
        {"role": "system", "content": "Classify the support ticket priority. Respond with only: Urgent, High, Medium, or Low."},
        {"role": "user", "content": item["ticket"]},
        {"role": "assistant", "content": item["priority"]}
    ]

def make_jsonl(items):
    """Convert items to JSONL string."""
    lines = [json.dumps({"messages": messages_for(item)}) for item in items]
    return "\n".join(lines)

def write_jsonl(items, filepath):
    """Write items to JSONL file."""
    os.makedirs(os.path.dirname(filepath) or ".", exist_ok=True)
    with open(filepath, "w", encoding="utf-8") as f:
        f.write(make_jsonl(items))

# Show example JSONL entry
print("Example JSONL entry:")
print(json.dumps({"messages": messages_for(train_data[0])}, indent=2))

Example JSONL entry:
{
  "messages": [
    {
      "role": "system",
      "content": "Classify the support ticket priority. Respond with only: Urgent, High, Medium, or Low."
    },
    {
      "role": "user",
      "content": "Could you add keyboard shortcuts for common actions? Power users would appreciate this productivity feature."
    },
    {
      "role": "assistant",
      "content": "Medium"
    }
  ]
}


In [ ]:
# Export to JSONL files
JSONL_DIR = "jsonl"
os.makedirs(JSONL_DIR, exist_ok=True)

write_jsonl(train_data, f"{JSONL_DIR}/fine_tune_train.jsonl")
write_jsonl(val_data, f"{JSONL_DIR}/fine_tune_validation.jsonl")

print(f"Exported training data to {JSONL_DIR}/")
print(f"  - fine_tune_train.jsonl: {len(train_data)} examples")
print(f"  - fine_tune_validation.jsonl: {len(val_data)} examples")
print(f"\nThese files can be used with:")
print(f"  - OpenAI fine-tuning API (platform.openai.com)")
print(f"  - Open-source fine-tuning tools (Hugging Face, Axolotl, etc.)")

## Generate More Training Data (Optional)

Use the LLM to generate additional synthetic tickets for each priority level.

In [8]:
def generate_tickets(n_per_priority: int = 3):
    """Generate additional synthetic tickets using LLM."""
    prompt = f"""Generate {n_per_priority} realistic customer support tickets for each priority level.

Priority definitions:
- Urgent: System down, security breach, data loss, production outage
- High: Major feature broken, payment issues, significant user impact  
- Medium: Feature request, minor bug, performance concern
- Low: General inquiry, documentation question, cosmetic issue

Reply with ONLY a JSON array, no other text:
[{{"ticket": "ticket text here", "priority": "Urgent"}}, ...]"""
    
    try:
        response = client.chat.completions.create(
            model=FRONTIER_MODEL,
            messages=[{"role": "user", "content": prompt}],
            max_tokens=2000,
            temperature=0.7
        )
        if response.choices:
            raw = (response.choices[0].message.content or "").strip()
            # Clean markdown code blocks if present
            raw = re.sub(r"^```(?:json)?\s*", "", raw).strip()
            raw = re.sub(r"\s*```$", "", raw).strip()
            
            generated = json.loads(raw)
            valid = [
                g for g in generated 
                if isinstance(g, dict) and g.get("priority") in PRIORITIES and g.get("ticket")
            ]
            return valid
    except Exception as e:
        print(f"Generation failed: {e}")
    return []

# Uncomment to generate more data:
# new_tickets = generate_tickets(3)
# print(f"Generated {len(new_tickets)} new tickets")
# TICKETS.extend(new_tickets)

## Gradio UI: Interactive Ticket Classifier

A user-friendly interface for:
- Classifying individual tickets
- Running batch evaluation
- Comparing predictions with true labels

In [9]:
# Priority color mapping for display
PRIORITY_COLORS = {
    "Urgent": "#dc3545",  # Red
    "High": "#fd7e14",    # Orange
    "Medium": "#ffc107",  # Yellow
    "Low": "#28a745"      # Green
}

def classify_ticket(ticket_text: str) -> str:
    """Classify a single ticket and return formatted result."""
    if not ticket_text.strip():
        return "Please enter a ticket description."
    
    priority = predict_baseline(ticket_text)
    color = PRIORITY_COLORS.get(priority, "#6c757d")
    
    return f"""## Classification Result

**Priority:** <span style="color:{color}; font-weight:bold; font-size:1.2em;">{priority}</span>

### Priority Definitions:
- **Urgent**: System down, security breach, data loss
- **High**: Major feature broken, payment issues
- **Medium**: Feature request, minor bug
- **Low**: General inquiry, documentation question"""

def run_evaluation() -> str:
    """Run evaluation on test set and return formatted results."""
    accuracy, results = calculate_accuracy(predict_baseline, test_data)
    
    # Build results table
    output = f"""## Evaluation Results

**Model:** {FRONTIER_MODEL}  
**Test Set Size:** {len(test_data)}  
**Accuracy:** {accuracy:.1%}

### Sample Predictions:

| Status | True | Predicted | Ticket |
|--------|------|-----------|--------|
"""
    
    for r in results[:10]:
        status = "OK" if r["correct"] else "WRONG"
        output += f"| {status} | {r['true']} | {r['predicted']} | {r['ticket'][:50]}... |\n"
    
    # Add confusion summary
    by_priority = {}
    for r in results:
        key = r["true"]
        if key not in by_priority:
            by_priority[key] = {"correct": 0, "total": 0}
        by_priority[key]["total"] += 1
        if r["correct"]:
            by_priority[key]["correct"] += 1
    
    output += "\n### Accuracy by Priority:\n\n"
    for priority in PRIORITIES:
        if priority in by_priority:
            stats = by_priority[priority]
            pct = stats["correct"] / stats["total"] * 100 if stats["total"] > 0 else 0
            output += f"- **{priority}**: {stats['correct']}/{stats['total']} ({pct:.0f}%)\n"
    
    return output

def get_sample_ticket(priority: str) -> str:
    """Get a random sample ticket for the selected priority."""
    matching = [t for t in TICKETS if t["priority"] == priority]
    if matching:
        return random.choice(matching)["ticket"]
    return ""

def export_data() -> str:
    """Show export status."""
    train_path = f"{JSONL_DIR}/fine_tune_train.jsonl"
    val_path = f"{JSONL_DIR}/fine_tune_validation.jsonl"
    
    train_exists = os.path.exists(train_path)
    val_exists = os.path.exists(val_path)
    
    return f"""## JSONL Export Status

| File | Status | Examples |
|------|--------|----------|
| {train_path} | {'Exists' if train_exists else 'Missing'} | {len(train_data)} |
| {val_path} | {'Exists' if val_exists else 'Missing'} | {len(val_data)} |

### Next Steps for Fine-Tuning:

1. **OpenAI Fine-Tuning:**
   ```python
   openai.files.create(file=open("{train_path}", "rb"), purpose="fine-tune")
   openai.fine_tuning.jobs.create(training_file=file_id, model="gpt-4.1-nano-2025-04-14")
   ```

2. **Open-Source Fine-Tuning:**
   - Use with Hugging Face Transformers
   - Compatible with Axolotl, LLaMA-Factory, etc.
"""

In [10]:
# Build Gradio Interface
with gr.Blocks(title="Support Ticket Classifier", theme=gr.themes.Soft()) as demo:
    gr.Markdown("""# Customer Support Ticket Priority Classifier
    
Classify support tickets into priority levels using a frontier LLM.
This demonstrates the baseline performance before fine-tuning.""")
    
    with gr.Tabs():
        # Tab 1: Single Ticket Classification
        with gr.Tab("Classify Ticket"):
            with gr.Row():
                with gr.Column():
                    ticket_input = gr.Textbox(
                        label="Ticket Description",
                        placeholder="Enter the support ticket text here...",
                        lines=5
                    )
                    with gr.Row():
                        classify_btn = gr.Button("Classify", variant="primary")
                        clear_btn = gr.Button("Clear")
                    
                    gr.Markdown("### Load Sample Ticket:")
                    with gr.Row():
                        for priority in PRIORITIES:
                            btn = gr.Button(priority, size="sm")
                            btn.click(
                                fn=lambda p=priority: get_sample_ticket(p),
                                outputs=ticket_input
                            )
                
                with gr.Column():
                    result_output = gr.Markdown(label="Result")
            
            classify_btn.click(fn=classify_ticket, inputs=ticket_input, outputs=result_output)
            clear_btn.click(fn=lambda: ("", ""), outputs=[ticket_input, result_output])
        
        # Tab 2: Batch Evaluation
        with gr.Tab("Evaluate Model"):
            gr.Markdown("""### Baseline Evaluation
            
Run the zero-shot classifier on the test set to measure baseline accuracy.
This is the number to beat with fine-tuning!""")
            
            eval_btn = gr.Button("Run Evaluation", variant="primary")
            eval_output = gr.Markdown()
            
            eval_btn.click(fn=run_evaluation, outputs=eval_output)
        
        # Tab 3: Export Data
        with gr.Tab("Export JSONL"):
            gr.Markdown("""### Training Data Export
            
View the exported JSONL files for fine-tuning.""")
            
            export_btn = gr.Button("Check Export Status", variant="primary")
            export_output = gr.Markdown()
            
            export_btn.click(fn=export_data, outputs=export_output)
        
        # Tab 4: Dataset Info
        with gr.Tab("Dataset Info"):
            dataset_info = f"""### Dataset Statistics

| Split | Count |
|-------|-------|
| Total | {len(TICKETS)} |
| Train | {len(train_data)} |
| Validation | {len(val_data)} |
| Test | {len(test_data)} |

### Priority Distribution (Full Dataset):

| Priority | Count | Percentage |
|----------|-------|------------|
"""
            counts = Counter(t['priority'] for t in TICKETS)
            for priority in PRIORITIES:
                count = counts.get(priority, 0)
                pct = count / len(TICKETS) * 100
                dataset_info += f"| {priority} | {count} | {pct:.1f}% |\n"
            
            dataset_info += f"""\n### Model Configuration

- **Model:** {FRONTIER_MODEL}
- **API:** {'OpenRouter' if 'openrouter' in OPENROUTER_BASE_URL else 'OpenAI'}
- **Temperature:** 0 (deterministic)
"""
            gr.Markdown(dataset_info)

# Launch the interface
demo.launch()

* Running on local URL:  http://127.0.0.1:7866
* To create a public link, set `share=True` in `launch()`.


Prediction error: Error code: 400 - {'error': {'message': 'Provider returned error', 'code': 400, 'metadata': {'raw': '{\n  "error": {\n    "message": "Invalid \'max_output_tokens\': integer below minimum value. Expected a value >= 16, but got 10 instead.",\n    "type": "invalid_request_error",\n    "param": "max_output_tokens",\n    "code": "integer_below_min_value"\n  }\n}', 'provider_name': 'Azure', 'is_byok': False}}, 'user_id': 'user_39l2mYQ8CncrKVDueHpsmmP8E63'}
Prediction error: Error code: 400 - {'error': {'message': 'Provider returned error', 'code': 400, 'metadata': {'raw': '{\n  "error": {\n    "message": "Invalid \'max_output_tokens\': integer below minimum value. Expected a value >= 16, but got 10 instead.",\n    "type": "invalid_request_error",\n    "param": "max_output_tokens",\n    "code": "integer_below_min_value"\n  }\n}', 'provider_name': 'Azure', 'is_byok': False}}, 'user_id': 'user_39l2mYQ8CncrKVDueHpsmmP8E63'}
Prediction error: Error code: 400 - {'error': {'messag

## Fine-Tuning with OpenAI (Optional)

If you have an OpenAI API key and want to fine-tune, use the exported JSONL files:

In [ ]:
# Uncomment and run this section to fine-tune with OpenAI
# Requires: OPENAI_API_KEY environment variable

'''
from openai import OpenAI

# Initialize OpenAI client (not OpenRouter)
openai_client = OpenAI()  # Uses OPENAI_API_KEY

# Upload training file
with open(f"{JSONL_DIR}/fine_tune_train.jsonl", "rb") as f:
    train_file = openai_client.files.create(file=f, purpose="fine-tune")
print(f"Training file uploaded: {train_file.id}")

# Upload validation file
with open(f"{JSONL_DIR}/fine_tune_validation.jsonl", "rb") as f:
    val_file = openai_client.files.create(file=f, purpose="fine-tune")
print(f"Validation file uploaded: {val_file.id}")

# Create fine-tuning job
job = openai_client.fine_tuning.jobs.create(
    training_file=train_file.id,
    validation_file=val_file.id,
    model="gpt-4.1-nano-2025-04-14",
    hyperparameters={"n_epochs": 3},
    suffix="ticket-classifier"
)
print(f"Fine-tuning job created: {job.id}")

# Check job status
# openai_client.fine_tuning.jobs.retrieve(job.id)
'''

## Summary

This exercise demonstrated:

1. **Data Preparation**: Creating a synthetic dataset for classification
2. **Train/Val/Test Split**: Proper data splitting for ML evaluation
3. **Zero-Shot Baseline**: Measuring frontier model performance without fine-tuning
4. **JSONL Export**: Preparing data for fine-tuning APIs
5. **Gradio UI**: Building an interactive classifier interface

### Key Metrics

- **Baseline Accuracy**: The zero-shot classifier's performance (your number to beat)
- **Target**: Fine-tuned model should exceed baseline accuracy

### Next Steps

1. Run fine-tuning with OpenAI or open-source tools
2. Compare fine-tuned model accuracy vs baseline
3. Experiment with:
   - More training data
   - Different base models
   - Hyperparameter tuning (epochs, batch size)